# Project: RAG Knowledge Base

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will build an agent that indexes documents, creates a retriever tool, and answers questions with cited sources.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Prepare Documents

Create a collection of text documents representing the knowledge base.

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Python's asyncio library provides infrastructure for writing single-threaded concurrent code using coroutines. It uses an event loop to manage async tasks efficiently.",
        metadata={"source": "python-async-guide", "topic": "python"},
    ),
    Document(
        page_content="REST APIs use HTTP methods like GET, POST, PUT, and DELETE to perform CRUD operations. They are stateless and use URLs to identify resources.",
        metadata={"source": "api-design-handbook", "topic": "apis"},
    ),
    Document(
        page_content="PostgreSQL is a powerful open-source relational database. It supports advanced features like JSON columns, full-text search, and window functions.",
        metadata={"source": "database-fundamentals", "topic": "databases"},
    ),
    Document(
        page_content="Git branching allows developers to work on features independently. The main branch contains production code, while feature branches isolate changes.",
        metadata={"source": "git-workflows", "topic": "git"},
    ),
    Document(
        page_content="Docker Compose defines multi-container applications in a YAML file. It manages networking, volumes, and dependencies between services.",
        metadata={"source": "docker-handbook", "topic": "devops"},
    ),
    Document(
        page_content="Unit tests verify individual functions in isolation. Use pytest fixtures for setup and parametrize for testing multiple inputs.",
        metadata={"source": "testing-best-practices", "topic": "testing"},
    ),
    Document(
        page_content="FastAPI automatically generates OpenAPI documentation from your Python type hints. It supports async endpoints, dependency injection, and Pydantic validation.",
        metadata={"source": "fastapi-tutorial", "topic": "python"},
    ),
    Document(
        page_content="Redis is an in-memory data store used for caching, session management, and message queuing. It supports data structures like strings, lists, sets, and sorted sets.",
        metadata={"source": "caching-strategies", "topic": "databases"},
    ),
]

print(f"Prepared {len(documents)} documents")

## 4. Create the Vector Store and Retriever

Index documents with embeddings and create a retriever for similarity search.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(documents)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("Documents indexed. Retriever ready.")

## 5. Build the Retriever Tool

Wrap the retriever in a `@tool` so the agent can search with cited sources.

In [ ]:
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """Search the technical knowledge base for information.

    Use this tool to find answers about Python, APIs, databases,
    Git, Docker, testing, and other technical topics.
    """
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant documents found."

    results = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        topic = doc.metadata.get("topic", "general")
        results.append(f"[Source: {source} | Topic: {topic}]\n{doc.page_content}")

    return "\n\n---\n\n".join(results)

print("Tool name:", search_knowledge_base.name)
print("Tool description:", search_knowledge_base.description)

## 6. Create the Agent

Build the agent with `create_react_agent` and the retriever tool.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

model = init_chat_model("gpt-4o-mini", model_provider="openai")

agent = create_react_agent(
    model=model,
    tools=[search_knowledge_base],
    prompt=(
        "You are a technical knowledge base assistant. "
        "Always search the knowledge base before answering questions. "
        "When you use information from the knowledge base, cite the source document. "
        "If the knowledge base doesn't have the answer, say so clearly."
    ),
)

## 7. Query the Knowledge Base

Ask questions and get answers with cited sources.

In [ ]:
from langchain_core.messages import HumanMessage

result = agent.invoke({
    "messages": [HumanMessage(content="How does Python handle async programming?")]
})
print(result["messages"][-1].content)

## 8. Multi-Topic Queries

The agent can search multiple times to answer complex questions.

In [ ]:
result = agent.invoke({
    "messages": [HumanMessage(content="Compare PostgreSQL and Redis — when would I use each?")]
})
print(result["messages"][-1].content)

## 9. Handling Missing Information

Ask about something not in the knowledge base.

In [ ]:
result = agent.invoke({
    "messages": [HumanMessage(content="How do I deploy to Kubernetes?")]
})
print(result["messages"][-1].content)

## 10. Full Workflow

Run multiple questions through the knowledge base agent.

In [ ]:
questions = [
    "How does Python handle async programming?",
    "Compare PostgreSQL and Redis.",
    "What's the best way to write unit tests?",
]

for question in questions:
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    print(f"Q: {question}")
    print(f"A: {result['messages'][-1].content}\n")

## Summary

- A RAG knowledge base agent combines document indexing with agent-based retrieval
- Wrap the retriever in a `@tool` with a clear docstring
- Include source metadata in tool output for citations
- The agent decides when to search and can issue multiple queries
- Use `InMemoryVectorStore` for prototyping; swap to a persistent store for production